# Quote / Tip of the Day Agent — cwait (Igniters Week 8)

This notebook implements a **Quote / Tip of the Day** agent: it (1) **fetches** quotes from a free public API, (2) **picks or rewrites** one into a short "tip of the day" using an LLM, and (3) **displays** the result in a Gradio UI. Use the **Refresh** button to get a new tip.

**Requirements:** `OPENAI_API_KEY` in your `.env` (or litellm env). Run from repo root or ensure `week8` is on `PYTHONPATH` so the course `Agent` base class can be imported.

In [1]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
week8_dir = notebook_dir.parent.parent
if (week8_dir / "agents").exists() and (week8_dir / "agents" / "agent.py").exists():
    os.chdir(week8_dir)
    sys.path.insert(0, str(week8_dir))
    print(f"Working directory: {os.getcwd()}")
else:
    week8_dir = notebook_dir
    for _ in range(5):
        week8_dir = week8_dir.parent
        if (week8_dir / "agents" / "agent.py").exists():
            os.chdir(week8_dir)
            sys.path.insert(0, str(week8_dir))
            print(f"Working directory: {os.getcwd()}")
            break
    else:
        print("Warning: week8 agents not found; Agent base class may be unavailable.")

from dotenv import load_dotenv
load_dotenv(override=True)

Working directory: /Users/collinewaitire/projects/python-p/llm_engineering/week8


True

## Quote Fetcher Agent

Fetches quotes from the [quotable.io](https://quotable.io) API (no key required). Returns a list of `{content, author}`. Uses the course `Agent` base class for colored logging.

In [4]:
import logging
import requests
from typing import List, Dict

logging.getLogger().setLevel(logging.INFO)

try:
    from agents.agent import Agent
except ImportError:
    class Agent:
        name = ""
        color = ""
        def log(self, msg): logging.info(f"[{self.name}] {msg}")


class QuoteFetcherAgent(Agent):
    name = "Quote Fetcher"
    color = Agent.CYAN
    API_URL = "https://api.quotable.io/quotes"
    FALLBACK_QUOTES = [
        {"content": "The only way to do great work is to love what you do.", "author": "Steve Jobs"},
        {"content": "Stay curious, stay humble.", "author": "Unknown"},
    ]

    def fetch(self, limit: int = 5) -> List[Dict[str, str]]:
        self.log("Fetching quotes from API")
        try:
            r = requests.get(f"{self.API_URL}?limit={limit}", timeout=10)
            r.raise_for_status()
            data = r.json()
            results = data.get("results", data) if isinstance(data, dict) else data
            if not isinstance(results, list):
                results = [data] if isinstance(data, dict) and "content" in data else []
            quotes = [{"content": q.get("content", ""), "author": q.get("author", "Unknown")} for q in results]
            if not quotes:
                quotes = self.FALLBACK_QUOTES
            self.log(f"Fetched {len(quotes)} quotes")
            return quotes
        except Exception as e:
            self.log(f"API error, using fallback: {e}")
            return self.FALLBACK_QUOTES

## Tip Picker Agent

Takes a list of quotes and uses the LLM to pick the most inspiring one and rephrase it as a single short "tip of the day" (1–2 sentences).

In [5]:
from openai import OpenAI


class TipPickerAgent(Agent):
    name = "Tip Picker"
    color = Agent.GREEN
    MODEL = "gpt-4o-mini"

    def __init__(self):
        self.client = OpenAI()

    def pick_tip(self, quotes: List[Dict[str, str]]) -> str:
        self.log("Choosing and rephrasing as tip of the day")
        if not quotes:
            return "Stay curious and keep learning."
        text = "\n".join([f"- {q['content']} (— {q['author']})" for q in quotes])
        sys_msg = (
            "Given the following quotes, choose the most inspiring one and rephrase it as a single "
            "tip of the day in 1–2 sentences. Reply with only the tip, no preamble or quotes."
        )
        resp = self.client.chat.completions.create(
            model=self.MODEL,
            messages=[
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": text},
            ],
            max_tokens=150,
        )
        tip = (resp.choices[0].message.content or "").strip()
        self.log("Tip ready")
        return tip or "Stay curious and keep learning."

## Pipeline: get_tip_of_the_day()

Orchestrates fetch → pick/rewrite → return the tip string.

In [6]:
def get_tip_of_the_day() -> str:
    fetcher = QuoteFetcherAgent()
    picker = TipPickerAgent()
    quotes = fetcher.fetch(limit=5)
    return picker.pick_tip(quotes)

## Run once (optional)

Verify the pipeline works before launching the UI.

In [7]:
tip = get_tip_of_the_day()
print(tip)

INFO:root:[Quote Fetcher] Fetching quotes from API
INFO:root:[Quote Fetcher] API error, using fallback: HTTPSConnectionPool(host='api.quotable.io', port=443): Max retries exceeded with url: /quotes?limit=5 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1010)')))
INFO:root:[Tip Picker] Choosing and rephrasing as tip of the day
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Tip Picker] Tip ready


Embrace your passions and stay curious; loving what you do combined with a humble approach can lead to great achievements.


## Gradio UI

Title, short description, and a **Refresh** button that runs the pipeline and updates the tip display.

In [ ]:
import gradio as gr

with gr.Blocks(title="Tip of the Day", fill_width=True) as ui:
    gr.Markdown('<div style="text-align: center; font-size: 24px;">Quote / Tip of the Day</div>')
    gr.Markdown('<div style="text-align: center; font-size: 14px;">Fetch quotes from the API, then the LLM picks one and rephrases it as a short tip. Click Refresh for a new tip.</div>')
    tip_display = gr.Markdown(value="Loading your tip of the day…")
    refresh_btn = gr.Button("Refresh")

    def on_refresh():
        return get_tip_of_the_day()

    refresh_btn.click(fn=on_refresh, inputs=[], outputs=[tip_display])
    ui.load(fn=on_refresh, inputs=[], outputs=[tip_display])

ui.launch(inbrowser=True)

INFO:httpx:HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


INFO:root:[Quote Fetcher] Fetching quotes from API
INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
INFO:root:[Quote Fetcher] API error, using fallback: HTTPSConnectionPool(host='api.quotable.io', port=443): Max retries exceeded with url: /quotes?limit=5 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1010)')))
INFO:root:[Tip Picker] Choosing and rephrasing as tip of the day
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Tip Picker] Tip ready
INFO:root:[Quote Fetcher] Fetching quotes from API
INFO:root:[Quote Fetcher] API error, using fallback: HTTPSConnectionPool(host='api.quotable.io', port=443): Max retries exceeded with url: /quotes?limit=5 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1010)')))
INFO:root:[Tip 